# XL430 pair

Apply the marked equation, then change one gain at a time.
Click the red **Stop** button shown by each trial to end the loop and turn torque off.
Plots appear automatically. The final `await` lets Jupyter receive button clicks.

## Connect

Set the serial port, baud and motor ID(s). `PWM_LIMIT=0.80` allows up to 80% PWM output.

In [ ]:
import dxl_utils as dxl

PWM_LIMIT = 0.80  # duty fraction; 0.80 = 80%
lab = dxl.connect("/dev/ttyACM0", station="A1", baud=1_000_000,
                  leader=1, follower=2, pwm_limit=PWM_LIMIT, stop_button=True, demo=False)

## $u_F=k_p(q_L-q_F)+k_d(\dot q_L-\dot q_F)$

The follower pulls towards the leader position; the velocity term damps their relative motion. Commands are PWM duty fractions. Raise the position gain to reduce tracking error, then add damping and compare overshoot.

**Change:** `kp`: 0.06 → 0.12 → 0.18; `kd`: 0 → 0.008. Then `T`: 0.001 → 0.005 → 0.010.

**Delay:** `delay_ms=0` delays outgoing commands; try 10, 50 and 100 ms with gains unchanged.

In [ ]:
T = 0.001  # seconds; then compare 0.005 and 0.010
#TODO: kp, kd = 0
delay_ms = 0.0
delay = dxl.Delay(samples=round(delay_ms/1000/T))
with lab.run("1_unilateral", leader="off", dt=T) as run:
    while run.active:
        L, F = run.read()
        #TODO: uF = 
        uF, = delay(uF)
        run.write(follower=uF)
        await run.wait_async()

## $u_L=-\beta\,\mathrm{PWM}_F$

The follower tracks hand position using its internal position controller. Reflect its actual PWM output to the leader: resistance should increase when the follower presses against an obstacle. PWM is an effort proxy, not a calibrated contact-force measurement.

**Change:** `beta`: 0.25 → 0.5 → 1.0. Compare free motion and contact.

**Delay:** `delay_ms=0` delays outgoing commands; try 10, 50 and 100 ms with gains unchanged.

In [ ]:
T = 0.001
beta = 0.5
delay_ms = 0.0
delay = dxl.Delay(samples=round(delay_ms/1000/T))
with lab.run("2_effort_feedback", leader="effort", follower="position", position_kp=150, dt=T) as run:
    while run.active:
        L, F = run.read()
        #TODO: uL = 
        uL, qsend = delay(uL, L.q)
        run.write(leader=uL, follower_position=qsend)
        await run.wait_async()

## Admittance: $\dot q_c=k_a(q_L-q_c)/b_a$

Push the leader away from its commanded position to make both motors move. Hand deflection acts as an input-effort proxy; this is not measured-force admittance. Both motors use powered position control, and the commanded motion is bounded to ±0.52 rad (30°).

**Change:** `damping`: 4 → 2 → 1. Lower damping gives more motion for the same push. Keep `gain` fixed; then compare `T`: 0.001 → 0.005 → 0.010.

**Delay:** `delay_ms=0` delays outgoing commands; try 10, 50 and 100 ms with gains unchanged.

In [ ]:
T = 0.001
gain, damping, limit = 1.0, 2.0, 0.52
delay_ms = 0.0
delay = dxl.Delay(samples=round(delay_ms/1000/T))
with lab.run("3_admittance", leader="position", follower="position", position_kp=100, dt=T) as run:
    qcmd = dxl.clip(run.leader.q, limit)
    while run.active:
        L, F = run.read()
        #TODO: velocity = 
        qcmd = dxl.clip(qcmd + run.dt*velocity, limit)
        qsend, = delay(qcmd)
        run.write(leader_position=qsend, follower_position=qsend, target=qcmd)
        await run.wait_async()

## Disconnect

In [ ]:
lab.close()